# 03 - Audio Data

# Purpose

Download Eurovision audio files for the years 2008–2026, strictly verify their integrity (existence, file size, duration), gracefully retry any failures, extract acoustic features using Essentia, and export a master feature dataset.

In [1]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Cell 2: Install everything
!pip install --upgrade yt-dlp
!pip install -U -q pandas==2.2.2 tqdm
!pip install -U -q essentia
!apt-get -qq update
!apt-get -qq install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.4 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
# Cell 3: Imports
import os
import re
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np
import yt_dlp
from tqdm.notebook import tqdm

import essentia.standard as es

# Disable Essentia warnings to keep terminal output clean
import warnings
warnings.filterwarnings('ignore')

In [4]:
# Cell 4: Load contestants.csv and filter 2008–2026
BASE_DIR = Path("/content/drive/MyDrive/Eurovision_Prediction")
DATASETS_DIR = BASE_DIR / "datasets"
AUDIO_ROOT = BASE_DIR / "audio"

CONTESTANTS_PATH = DATASETS_DIR / "contestants.csv"

# Load and clean data
contestants = (
    pd.read_csv(CONTESTANTS_PATH)
      .query("2008 <= year <= 2026")
      .drop_duplicates(subset=["year", "to_country_id", "song", "performer"])
      .reset_index(drop=True)
)

print(f"Total unique songs to process (2008-2026): {len(contestants)}")

Total unique songs to process (2008-2026): 718


In [5]:
# Cell 5: Create audio folders
for year in range(2008, 2027):
    (AUDIO_ROOT / str(year)).mkdir(parents=True, exist_ok=True)

print(f"Audio directory structure established at: {AUDIO_ROOT}")

Audio directory structure established at: /content/drive/MyDrive/Eurovision_Prediction/audio


In [8]:
import re
# Assuming yt_dlp, Path, tqdm, contestants, and AUDIO_ROOT are defined earlier in your notebook

# Cell 6: Download audio
def clean(text):
    text = str(text)
    text = re.sub(r'[\\/*?:"<>|]', "", text)  # Remove illegal characters for filenames
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def get_expected_filepath(year, country_id, song, performer):
    year = int(year)
    country = clean(country_id).upper()
    performer_clean = clean(performer)
    song_clean = clean(song)
    return AUDIO_ROOT / str(year) / f"{country}_{song_clean}_{performer_clean}.mp3"

def download_song(url, filepath):
    if filepath.exists():
        return True

    output_template = str(filepath).replace(".mp3", ".%(ext)s")

    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": output_template,
        "extractaudio": True,
        "audioformat": "mp3",
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": "mp3",
            "preferredquality": "320",
        }],
        "noplaylist": True,
        "quiet": True,
        "no_warnings": True,
        "ignoreerrors": True,

        # 🔑 Authenticate via cookies to bypass YouTube bot detection
        "cookiefile": "cookies.txt",
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        return True
    except Exception:
        return False

print("Starting initial download pass...")
for _, row in tqdm(contestants.iterrows(), total=len(contestants)):
    filepath = get_expected_filepath(row["year"], row["to_country_id"], row["song"], row["performer"])
    download_song(row["youtube_url"], filepath)

Starting initial download pass...


  0%|          | 0/718 [00:00<?, ?it/s]

In [9]:
# Cell 7: Verify downloads & create manifest
def get_audio_duration(filepath):
    """Uses ffprobe to instantly get the duration without loading the audio into RAM."""
    try:
        result = subprocess.run(
            ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", str(filepath)],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )
        return float(result.stdout.strip())
    except Exception:
        return 0.0

manifest_rows = []

print("Verifying downloads strictly (existence, size, duration)...")
for _, row in tqdm(contestants.iterrows(), total=len(contestants)):
    filepath = get_expected_filepath(row["year"], row["to_country_id"], row["song"], row["performer"])

    exists = filepath.exists()
    size_bytes = filepath.stat().st_size if exists else 0
    duration = get_audio_duration(filepath) if exists and size_bytes > 0 else 0.0

    # Validation criteria
    is_valid = exists and (size_bytes > 500_000) and (duration > 30.0)

    # If it exists but is corrupted/too short, delete it to force a clean redownload
    if exists and not is_valid:
        filepath.unlink()
        exists = False

    manifest_rows.append({
        "year": row["year"],
        "country": row["to_country_id"],
        "performer": row["performer"],
        "song": row["song"],
        "youtube_url": row["youtube_url"],
        "filepath": str(filepath),
        "exists": exists,
        "duration": duration,
        "is_valid": is_valid
    })

manifest = pd.DataFrame(manifest_rows)
MANIFEST_PATH = DATASETS_DIR / "audio_manifest.csv"
manifest.to_csv(MANIFEST_PATH, index=False)
print(f"Manifest created successfully at: {MANIFEST_PATH}")

Verifying downloads strictly (existence, size, duration)...


  0%|          | 0/718 [00:00<?, ?it/s]

Manifest created successfully at: /content/drive/MyDrive/Eurovision_Prediction/datasets/audio_manifest.csv


In [10]:
# Cell 8: Retry failed downloads
failed_downloads = manifest[~manifest["is_valid"]].copy()

if len(failed_downloads) > 0:
    print(f"Retrying {len(failed_downloads)} failed or corrupted downloads...")

    for idx, row in tqdm(failed_downloads.iterrows(), total=len(failed_downloads)):
        filepath = Path(row["filepath"])
        success = download_song(row["youtube_url"], filepath)

        if success:
            # Re-verify
            exists = filepath.exists()
            size_bytes = filepath.stat().st_size if exists else 0
            duration = get_audio_duration(filepath) if exists and size_bytes > 0 else 0.0
            is_valid = exists and (size_bytes > 500_000) and (duration > 30.0)

            # Update manifest dataframe inline
            manifest.at[idx, "exists"] = exists
            manifest.at[idx, "duration"] = duration
            manifest.at[idx, "is_valid"] = is_valid

    # Save updated manifest
    manifest.to_csv(MANIFEST_PATH, index=False)
    print("Retry pass complete. Manifest updated.")
else:
    print("No failed downloads to retry. All good!")

Retrying 1 failed or corrupted downloads...


  0%|          | 0/1 [00:00<?, ?it/s]

Retry pass complete. Manifest updated.


In [11]:
# Cell 9: Final verification report
expected = len(manifest)
successful = manifest["is_valid"].sum()
missing = expected - successful
success_rate = (successful / expected) * 100

print("-" * 40)
print("FINAL DOWNLOAD REPORT")
print("-" * 40)
print(f"Expected songs:   {expected}")
print(f"Successful:       {successful}")
print(f"Missing/Failed:   {missing}")
print(f"Success Rate:     {success_rate:.2f}%")
print("-" * 40)

if missing > 0:
    print("\nSongs still missing after retry:")
    display(manifest[~manifest["is_valid"]][["year", "country", "performer", "song"]])

----------------------------------------
FINAL DOWNLOAD REPORT
----------------------------------------
Expected songs:   718
Successful:       718
Missing/Failed:   0
Success Rate:     100.00%
----------------------------------------


In [12]:
# Cell 10: Extract Essentia features
# Initialize the high-level MusicExtractor
extractor = es.MusicExtractor(
    lowlevelStats=['mean', 'stdev'],
    rhythmStats=['mean', 'stdev'],
    tonalStats=['mean', 'stdev']
)

valid_songs = manifest[manifest["is_valid"]].copy()
feature_rows = []

print("Extracting Essentia features (this will take some time)...")
for _, row in tqdm(valid_songs.iterrows(), total=len(valid_songs)):
    try:
        # Compute features
        features, _ = extractor(row["filepath"])

        # Flatten the Essentia Pool object into a dictionary
        # We only want scalar numerical values (int/float)
        song_features = {
            "year": row["year"],
            "country": row["country"]
        }

        for key in features.descriptorNames():
            val = features[key]
            # Ensure it's a single numerical value, not an array
            if isinstance(val, (int, float, np.float32, np.float64)):
                # Clean up nested Essentia keys for pandas column naming
                clean_key = key.replace(".", "_")
                song_features[clean_key] = float(val)

        feature_rows.append(song_features)

    except Exception as e:
        print(f"Failed to extract features for {row['filepath']}: {e}")

features_df = pd.DataFrame(feature_rows)

Extracting Essentia features (this will take some time)...


  0%|          | 0/718 [00:00<?, ?it/s]

In [13]:
# Cell 11: Save audio_features.csv
OUTPUT_FEATURES = DATASETS_DIR / "audio_features.csv"

features_df.to_csv(OUTPUT_FEATURES, index=False)

print(f"Dataset successfully saved to: {OUTPUT_FEATURES}")
print(f"Total shape: {features_df.shape}")

Dataset successfully saved to: /content/drive/MyDrive/Eurovision_Prediction/datasets/audio_features.csv
Total shape: (718, 122)


In [14]:
# Cell 12: Summary statistics
print("--- Dataframe Info ---")
features_df.info()

print("\n--- Summary Statistics ---")
display(features_df.describe())

print("\n--- Missing Values Check ---")
missing_counts = features_df.isnull().sum()
if missing_counts.sum() == 0:
    print("No missing values found in the extracted features.")
else:
    print(f"Found missing values in columns:\n{missing_counts[missing_counts > 0]}")

print("\n--- Data Sample ---")
display(features_df.head())

--- Dataframe Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 718 entries, 0 to 717
Columns: 122 entries, year to tonal_tuning_nontempered_energy_ratio
dtypes: float64(120), int64(1), object(1)
memory usage: 684.5+ KB

--- Summary Statistics ---


,year,lowlevel_average_loudness,lowlevel_barkbands_crest_mean,lowlevel_barkbands_crest_stdev,lowlevel_barkbands_flatness_db_mean,lowlevel_barkbands_flatness_db_stdev,lowlevel_barkbands_kurtosis_mean,lowlevel_barkbands_kurtosis_stdev,lowlevel_barkbands_skewness_mean,lowlevel_barkbands_skewness_stdev,...,tonal_hpcp_crest_stdev,tonal_hpcp_entropy_mean,tonal_hpcp_entropy_stdev,tonal_key_edma_strength,tonal_key_krumhansl_strength,tonal_key_temperley_strength,tonal_tuning_diatonic_strength,tonal_tuning_equal_tempered_deviation,tonal_tuning_frequency,tonal_tuning_nontempered_energy_ratio
count,718.000000,718.000000,718.000000,718.000000,718.000000,718.000000,718.000000,718.000000,718.000000,718.000000,...,718.000000,718.000000,718.000000,718.000000,718.000000,718.000000,718.000000,718.000000,718.000000,718.000000
mean,2016.610028,0.712754,10.825680,4.631951,0.149036,0.060294,5.099278,16.518801,1.397982,1.419513,...,6.972773,1.960097,0.789521,0.706164,0.711836,0.715608,0.628388,0.194878,435.647086,0.887732
std,5.527768,0.233744,1.126513,0.575449,0.030092,0.015525,2.999988,35.342300,0.348124,0.383264,...,0.837037,0.169608,0.067747,0.097773,0.098392,0.098923,0.088913,0.083275,2.913706,0.076903
min,2008.000000,0.012437,7.405240,3.196738,0.075566,0.029503,-0.064934,2.056793,0.554499,0.665499,...,5.112980,1.486458,0.648501,0.271650,0.272046,0.293331,0.181910,0.000000,433.191071,0.666949
25%,2012.000000,0.583949,10.070977,4.223208,0.128720,0.049408,3.076225,6.818956,1.162383,1.146568,...,6.368717,1.851669,0.735861,0.643672,0.651360,0.653592,0.579637,0.190051,434.193115,0.894459
50%,2016.000000,0.796986,10.820937,4.622016,0.145861,0.057136,4.301392,10.062632,1.366397,1.344936,...,6.880739,1.958941,0.780028,0.712497,0.715901,0.721019,0.633823,0.222005,434.193115,0.917533
75%,2022.000000,0.892133,11.562820,5.001252,0.166666,0.067289,6.535338,16.695226,1.589139,1.649511,...,7.470803,2.071486,0.836090,0.776055,0.785101,0.787212,0.689522,0.245299,434.193115,0.933851
max,2026.000000,0.977911,15.004841,6.428306,0.294002,0.122232,20.958420,599.756042,2.759917,3.190375,...,9.833890,2.479495,0.998917,0.924652,0.927391,0.934005,0.876926,0.413384,453.678467,0.981571



--- Missing Values Check ---
No missing values found in the extracted features.

--- Data Sample ---


,year,country,lowlevel_average_loudness,lowlevel_barkbands_crest_mean,lowlevel_barkbands_crest_stdev,lowlevel_barkbands_flatness_db_mean,lowlevel_barkbands_flatness_db_stdev,lowlevel_barkbands_kurtosis_mean,lowlevel_barkbands_kurtosis_stdev,lowlevel_barkbands_skewness_mean,...,tonal_hpcp_crest_stdev,tonal_hpcp_entropy_mean,tonal_hpcp_entropy_stdev,tonal_key_edma_strength,tonal_key_krumhansl_strength,tonal_key_temperley_strength,tonal_tuning_diatonic_strength,tonal_tuning_equal_tempered_deviation,tonal_tuning_frequency,tonal_tuning_nontempered_energy_ratio
0,2008,ru,0.738203,10.060263,4.592516,0.132153,0.064652,2.955777,9.625051,1.206842,...,6.949013,1.940445,0.757875,0.582378,0.587510,0.590944,0.548034,0.075344,441.527557,0.790850
1,2008,ua,0.824829,10.227052,4.765164,0.123017,0.052716,2.000741,5.755677,0.986042,...,7.109819,2.111026,0.819952,0.602570,0.602628,0.600476,0.568076,0.202792,434.193115,0.907202
2,2008,gr,0.924662,9.128140,4.234442,0.094102,0.049245,1.978869,5.568943,1.123128,...,7.205321,2.103727,0.795173,0.694669,0.708951,0.702787,0.664536,0.288654,434.193115,0.938774
3,2008,am,0.698567,10.368389,4.976676,0.120196,0.047224,1.431407,4.790052,0.855768,...,6.394502,1.997572,0.761931,0.758539,0.766537,0.770383,0.681201,0.246661,434.193115,0.916581
4,2008,no,0.926753,9.151514,3.820127,0.123043,0.048714,1.210380,3.547405,0.899162,...,6.461064,1.915080,0.704693,0.800168,0.815495,0.827038,0.724398,0.059591,440.763123,0.740128
